In [ ]:
## S1 - S2: Setup environment + Load Data
# [1]
import pandas as pd, numpy as np
import xgboost as xgb
import shap
import pandas as pd
print("Environment OK")
def section(title):
    print(f"\n{'='*60}\n{title}\n")
def endSection():
    print(f"\n{'='*60}\n")
def tit(title):
    print(f"\n{title}\n")

# [2]
df = pd.read_csv('../data/paysim.csv')
print(f"Shape: {df.shape}")

# [3]: Ten 11 cot, dtype tung cot, va bo nho su dung.
section("[3] Data Understanding")
df.info()
endSection()
# [4]: Hien thi 10 dong dau
section("[4] Sample data")
print(df.head(10))
endSection()
# [5]: Kiem tra gia tri null
section("[5] Check null values")
print(df.isnull().sum())
endSection()
# [6]: Kiem tra duplicate
section("[6] Check duplicate values")
print(df.duplicated().sum())
endSection()
# [7]: Thong ke mo ta co ban
section("[7] Descriptive statistics")
print(df.describe())
endSection()
# [8]: Kiem tra gia tri duy nhat cua cac cot phan loai
section("[8] Unique values of categorical columns")
print(df['type'].value_counts())
print(df['isFraud'].value_counts())
print(df['isFlaggedFraud'].value_counts())
endSection()
# [9]: Verify y nghia step
section("[9] Verify step")
print(df['step'].min(), df['step'].max())
endSection()

# Verify fraud chi xay ra o TRANSFER va CASH_OUT
section("[10] Verify fraud only occurs in TRANSFER and CASH_OUT")
print(df[df['isFraud'] == 1]['type'].value_counts())
endSection()

# Verify cong thuc so du co khop logic khong
# Kiem tra vai dong xem oldbalanceOrg - amount co gan bang newbalanceOrig khong
# Muc dich: hieu vi sao can tao errorBalanceOrig. Nhieu dong so du khong khop hoan toan voi phep tru don gian, va do lech nay co the la tin hieu huu ich de phat hien gian lan.
section("[10] Balance formula check")
print(df[['oldbalanceOrg', 'amount', 'newbalanceOrig']].head(10))
endSection()

# Thu vai giao dich fraud that de co cam giac truc quan
# Doc bang mat vai dong de thay pattern rut can tai khoan: newbalanceOrig thuong gan bang 0 sau giao dich.
section("[11] Fraud sample")
print(df[df['isFraud'] == 1].head(10))
endSection()

In [ ]:
## S3: EDA - Class imbalance + Transaction type

# [10]: Phan tich class imbalance
section("[10] Class imbalance analysis")
tit("10.1: Tinh ty le chinh xac")
fraud_counts = df['isFraud'].value_counts()
fraud_pct = df['isFraud'].value_counts(normalize=True) * 100
print(fraud_counts)
print(fraud_pct)

tit("10.2: Ve bieu do (bar chart, log scale)")
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(6, 5))
colors = ['#4C72B0', '#C44E52']
bars = ax.bar(['Khong gian lan (0)', 'Gian lan (1)'], fraud_counts.values, color=colors)
ax.set_yscale('log')
ax.set_ylabel('So luong giao dich (log scale)')
ax.set_title('Phan phoi lop isFraud (mat can bang cuc doan)')

# Them so lieu that len dau moi cot cho de doc
for bar, count in zip(bars, fraud_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{count:,}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/class_imbalance.png', dpi=150)
plt.show()

# [11]: Phan tich theo transaction type
# 11.1: Crosstab tong quat
ct = pd.crosstab(df['type'], df['isFraud'])
ct.columns = ['Khong gian lan', 'Gian lan']
print(ct)

# 11.2: Tinh ty le fraud theo tung type
fraud_rate_by_type = df.groupby('type')['isFraud'].mean() * 100
print(fraud_rate_by_type.sort_values(ascending=False))

# 11.3: Ve bieu do ty le fraud theo type
fig, ax = plt.subplots(figsize=(7, 5))
fraud_rate_by_type.sort_values(ascending=False).plot(kind='bar', ax=ax, color='#C44E52')
ax.set_ylabel('Ty le gian lan (%)')
ax.set_xlabel('Loai giao dich')
ax.set_title('Ty le gian lan theo loai giao dich')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/fraud_rate_by_type.png', dpi=150)
plt.show()

# 11.4: Bieu do phan phoi tong the transaction type
fig, ax = plt.subplots(figsize=(7, 5))
df['type'].value_counts().plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_ylabel('So luong giao dich')
ax.set_title('Phan phoi so luong giao dich theo loai')
plt.tight_layout()
plt.savefig('../reports/figures/transaction_type_distribution.png', dpi=150)
plt.show()


# [12]: Phan phoi amount - fraud va non-fraud
# 12.1: Thong ke mo ta so sanh
print(df[df['isFraud']==0]['amount'].describe())
print(df[df['isFraud']==1]['amount'].describe())

# 12.2: Ve boxplot (log scale)
fig, ax = plt.subplots(figsize=(7, 5))
# Loc chi TRANSFER va CASH_OUT vi day la noi fraud that su xay ra
subset = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])]
sns.boxplot(data=subset, x='isFraud', y='amount', ax=ax)
ax.set_yscale('log')
ax.set_xticklabels(['Khong gian lan', 'Gian lan'])
ax.set_ylabel('So tien giao dich (log scale)')
ax.set_title('Phan phoi so tien giao dich theo trang thai gian lan\n(chi TRANSFER va CASH_OUT)')

plt.tight_layout()
plt.savefig('../reports/figures/amount_distribution_fraud.png', dpi=150)
plt.show()